# 05 — Typed Pipeline Integration

Phase 5 moves Pandera from explicit schema calls into function contracts.

We will explore:

- `DataFrame[Schema]`,
- `@pa.check_types(lazy=True)`,
- different input and output schemas,
- semantic output validation,
- end-to-end pipeline orchestration,
- source-data failures vs transformation defects.

## 1. Project setup

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab')

In [2]:
import pandas as pd
import pandera.pandas as pa
from pandera.typing import DataFrame

from pandera_lab import enrich_orders, run_order_pipeline
from pandera_lab.schemas import EnrichedOrderSchema, OrderSchema

## 2. Compare the input and output contracts

`OrderSchema` is the trusted order contract from Phase 4.

`EnrichedOrderSchema` inherits it and adds analytics fields.

In [3]:
OrderSchema.to_schema()

<Schema DataFrameSchema(columns={'order_id': <Schema Column(name=order_id, type=DataType(int64))>, 'customer_id': <Schema Column(name=customer_id, type=DataType(str))>, 'product_id': <Schema Column(name=product_id, type=DataType(str))>, 'quantity': <Schema Column(name=quantity, type=DataType(int64))>, 'unit_price': <Schema Column(name=unit_price, type=DataType(float64))>, 'discount': <Schema Column(name=discount, type=DataType(float64))>, 'total': <Schema Column(name=total, type=DataType(float64))>, 'status': <Schema Column(name=status, type=DataType(str))>, 'order_date': <Schema Column(name=order_date, type=DataType(datetime64[ns]))>}, checks=[<Check total_matches_formula>], parsers=[], index=None, dtype=None, coerce=False, strict=filter, name=OrderSchema, ordered=False, unique=None, report_duplicates=all, unique_column_names=False, add_missing_columns=False, title=None, description=Trusted analytical contract for an order record., metadata=None, drop_invalid_rows=False)>

In [4]:
EnrichedOrderSchema.to_schema()

<Schema DataFrameSchema(columns={'order_id': <Schema Column(name=order_id, type=DataType(int64))>, 'customer_id': <Schema Column(name=customer_id, type=DataType(str))>, 'product_id': <Schema Column(name=product_id, type=DataType(str))>, 'quantity': <Schema Column(name=quantity, type=DataType(int64))>, 'unit_price': <Schema Column(name=unit_price, type=DataType(float64))>, 'discount': <Schema Column(name=discount, type=DataType(float64))>, 'total': <Schema Column(name=total, type=DataType(float64))>, 'status': <Schema Column(name=status, type=DataType(str))>, 'order_date': <Schema Column(name=order_date, type=DataType(datetime64[ns]))>, 'gross_amount': <Schema Column(name=gross_amount, type=DataType(float64))>, 'discount_amount': <Schema Column(name=discount_amount, type=DataType(float64))>, 'net_amount': <Schema Column(name=net_amount, type=DataType(float64))>, 'order_month': <Schema Column(name=order_month, type=DataType(str))>, 'is_discounted': <Schema Column(name=is_discounted, type

## 3. Build realistic raw-like valid input

The values arrive as CSV-like strings and include a source-only column.

In [5]:
raw_like = pd.DataFrame({
    "order_id": ["5001", "5002", "5003"],
    "customer_id": ["C501", "C502", "C503"],
    "product_id": ["P501", "P502", "P503"],
    "quantity": ["2", "3", "1"],
    "unit_price": ["100.0", "20.0", "80.0"],
    "discount": ["0.10", "0.00", "0.25"],
    "total": ["180.0", "60.0", "60.0"],
    "status": ["paid", "shipped", "pending"],
    "order_date": ["2026-08-20", "2026-08-21", "2026-09-01"],
    "internal_note": ["source-a", "source-b", "source-c"],
})

raw_like.dtypes

order_id         object
customer_id      object
product_id       object
quantity         object
unit_price       object
discount         object
total            object
status           object
order_date       object
internal_note    object
dtype: object

## 4. Call the typed transformation directly

The function signature is:

```python
@pa.check_types(lazy=True)
def enrich_orders(
    df: DataFrame[OrderSchema],
) -> DataFrame[EnrichedOrderSchema]:
    ...
```

In [6]:
original_columns = raw_like.columns.tolist()
enriched = enrich_orders(raw_like)

enriched

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date,gross_amount,discount_amount,net_amount,order_month,is_discounted
0,5001,C501,P501,2,100.0,0.10,180.0,paid,2026-08-20,200.0,20.0,180.0,2026-08,True
1,5002,C502,P502,3,20.0,0.00,60.0,shipped,2026-08-21,60.0,0.0,60.0,2026-08,False
2,5003,C503,P503,1,80.0,0.25,60.0,pending,2026-09-01,80.0,20.0,60.0,2026-09,True


In [7]:
enriched.dtypes

order_id                    int64
customer_id                object
product_id                 object
quantity                    int64
unit_price                float64
discount                  float64
total                     float64
status                     object
order_date         datetime64[ns]
gross_amount              float64
discount_amount           float64
net_amount                float64
order_month                object
is_discounted                bool
dtype: object

Notice:

- numeric/date input fields were validated/coerced through `OrderSchema`,
- `internal_note` was filtered at the input contract,
- derived columns were added,
- the returned dataframe passed `EnrichedOrderSchema` before reaching us.

In [8]:
print("caller columns unchanged:", raw_like.columns.tolist() == original_columns)
print("returned columns:", enriched.columns.tolist())

caller columns unchanged: True
returned columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'total', 'status', 'order_date', 'gross_amount', 'discount_amount', 'net_amount', 'order_month', 'is_discounted']


## 5. Invalid input is stopped at the function boundary

In [9]:
bad_input = raw_like.copy()
bad_input.loc[0, "total"] = "999.0"

try:
    enrich_orders(bad_input)
except pa.errors.SchemaErrors as exc:
    input_error = exc
    print(type(exc).__name__)
    display(exc.failure_cases)

SchemaErrors


,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,order_id,total_matches_formula,0,5001,0
1,DataFrameSchema,customer_id,total_matches_formula,0,C501,0
2,DataFrameSchema,product_id,total_matches_formula,0,P501,0
3,DataFrameSchema,quantity,total_matches_formula,0,2,0
4,DataFrameSchema,unit_price,total_matches_formula,0,100.0,0
5,DataFrameSchema,discount,total_matches_formula,0,0.1,0
6,DataFrameSchema,total,total_matches_formula,0,999.0,0
7,DataFrameSchema,status,total_matches_formula,0,paid,0
8,DataFrameSchema,order_date,total_matches_formula,0,2026-08-20 00:00:00,0


The transformation should not produce a trusted result from an input that violates the Phase-4 order contract.

## 6. A broken output can fail even when input is valid

In [10]:
@pa.check_types(lazy=True)
def broken_output(
    df: DataFrame[OrderSchema],
) -> DataFrame[EnrichedOrderSchema]:
    gross = df["unit_price"] * df["quantity"]
    return df.assign(
        gross_amount=gross,
        discount_amount=gross * df["discount"],
        net_amount=999.0,  # deliberately wrong but still float
        order_month=df["order_date"].dt.strftime("%Y-%m"),
        is_discounted=df["discount"].gt(0),
    )

try:
    broken_output(raw_like)
except pa.errors.SchemaErrors as exc:
    output_error = exc
    display(exc.failure_cases)

,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,order_id,net_amount_matches_total,2,5001,0
31,DataFrameSchema,discount_amount,net_amount_matches_total,2,0.0,1
23,DataFrameSchema,status,net_amount_matches_total,2,pending,2
24,DataFrameSchema,order_date,net_amount_matches_total,2,2026-08-20 00:00:00,0
25,DataFrameSchema,order_date,net_amount_matches_total,2,2026-08-21 00:00:00,1
26,DataFrameSchema,order_date,net_amount_matches_total,2,2026-09-01 00:00:00,2
27,DataFrameSchema,gross_amount,net_amount_matches_total,2,200.0,0
28,DataFrameSchema,gross_amount,net_amount_matches_total,2,60.0,1
29,DataFrameSchema,gross_amount,net_amount_matches_total,2,80.0,2
30,DataFrameSchema,discount_amount,net_amount_matches_total,2,20.0,0


This is the key Phase-5 idea:

```text
valid input does not guarantee valid transformation output
```

The return annotation creates a runtime postcondition.

## 7. Missing output fields also fail

In [11]:
@pa.check_types(lazy=True)
def incomplete_output(
    df: DataFrame[OrderSchema],
) -> DataFrame[EnrichedOrderSchema]:
    gross = df["unit_price"] * df["quantity"]
    return df.assign(
        gross_amount=gross,
        discount_amount=gross * df["discount"],
        net_amount=df["total"],
        order_month=df["order_date"].dt.strftime("%Y-%m"),
        # is_discounted omitted deliberately
    )

try:
    incomplete_output(raw_like)
except pa.errors.SchemaErrors as exc:
    display(exc.failure_cases)

,column,failure_case,index,schema_context,check,check_number
0,EnrichedOrderSchema,is_discounted,None,DataFrameSchema,column_in_dataframe,None


## 8. Why keep `validate_orders` if the decorator validates input?

The two layers have different responsibilities:

```text
validate_orders
    -> untrusted batch diagnostics
    -> failure_cases
    -> operational reports

@check_types
    -> internal function contract
    -> developer guardrail
    -> input + output runtime validation
```

## 9. Run the full pipeline on the valid reference batch

In [12]:
valid_result = run_order_pipeline(
    input_path=ROOT / "data" / "reference" / "orders_valid.csv",
    output_path=ROOT / "data" / "clean" / "phase5_orders_enriched.csv",
    report_dir=ROOT / "reports",
)

valid_result

PipelineResult(succeeded=True, input_path=WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab/data/reference/orders_valid.csv'), output_path=WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab/data/clean/phase5_orders_enriched.csv'), detail_report_path=None, summary_report_path=None, rows_read=5, rows_written=5)

In [13]:
valid_output = pd.read_csv(valid_result.output_path)
valid_output

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date,gross_amount,discount_amount,net_amount,order_month,is_discounted
0,2001,C101,P101,2,100.0,0.10,180.0,paid,2026-08-01,200.0,20.0,180.0,2026-08,True
1,2002,C102,P102,1,50.0,0.00,50.0,pending,2026-08-02,50.0,0.0,50.0,2026-08,False
2,2003,C103,P103,3,20.0,0.25,45.0,shipped,2026-08-03,60.0,15.0,45.0,2026-08,True
3,2004,C104,P104,1,250.0,0.20,200.0,cancelled,2026-08-04,250.0,50.0,200.0,2026-08,True
4,2005,C105,P105,4,15.0,0.00,60.0,paid,2026-08-05,60.0,0.0,60.0,2026-08,False


## 10. Run the same pipeline on the deliberately invalid raw batch

In [14]:
invalid_result = run_order_pipeline(
    input_path=ROOT / "data" / "raw" / "orders.csv",
    output_path=ROOT / "data" / "clean" / "phase5_raw_orders_enriched.csv",
    report_dir=ROOT / "reports",
)

invalid_result

PipelineResult(succeeded=False, input_path=WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab/data/raw/orders.csv'), output_path=None, detail_report_path=WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab/reports/orders_validation_errors.csv'), summary_report_path=WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab/reports/orders_validation_summary.csv'), rows_read=15, rows_written=0)

In [15]:
if invalid_result.detail_report_path:
    detail = pd.read_csv(invalid_result.detail_report_path)
    display(detail)

if invalid_result.summary_report_path:
    summary = pd.read_csv(invalid_result.summary_report_path)
    display(summary)

,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,order_date,total_matches_formula,0.0,2026-08-06,5.0
1,DataFrameSchema,status,total_matches_formula,0.0,shipped,5.0
2,DataFrameSchema,total,total_matches_formula,0.0,100.0,5.0
3,DataFrameSchema,discount,total_matches_formula,0.0,0.1,5.0
4,DataFrameSchema,unit_price,total_matches_formula,0.0,75.0,5.0
5,DataFrameSchema,quantity,total_matches_formula,0.0,2,5.0
6,DataFrameSchema,product_id,total_matches_formula,0.0,P006,5.0
7,DataFrameSchema,customer_id,total_matches_formula,0.0,C006,5.0
8,DataFrameSchema,order_id,total_matches_formula,0.0,1006,5.0
9,Column,quantity,coerce_dtype('int64'),NaN,NaN,9.0


,column,check,failures
0,customer_id,not_nullable,1
1,customer_id,total_matches_formula,1
2,discount,greater_than_or_equal_to(0),1
3,discount,less_than_or_equal_to(1),1
4,discount,total_matches_formula,1
5,order_date,coerce_dtype('datetime64[ns]'),1
6,order_date,dtype('datetime64[ns]'),1
7,order_date,total_matches_formula,1
8,order_id,field_uniqueness,2
9,order_id,total_matches_formula,1


The invalid source batch creates reports and no trusted output.

That is intentionally different from a transformation contract exception, which is allowed to raise loudly.

## 11. Persistence creates another future boundary

The valid output was written to CSV.

Re-read it and inspect dtypes:

In [16]:
reloaded = pd.read_csv(valid_result.output_path)
reloaded.dtypes

order_id             int64
customer_id         object
product_id          object
quantity             int64
unit_price         float64
discount           float64
total              float64
status              object
order_date          object
gross_amount       float64
discount_amount    float64
net_amount         float64
order_month         object
is_discounted         bool
dtype: object

A validated in-memory dataframe and a later CSV re-read are not the same trust state.

If this file enters another system tomorrow, validate it again at that system's trust boundary.

# Phase-5 checkpoint

You should now be able to explain:

1. `DataFrame[Schema]`,
2. `@pa.check_types`,
3. runtime input and output validation,
4. different input/output schemas,
5. semantic output postconditions,
6. source-data failures vs transformation defects,
7. why the pipeline still has a dedicated batch validation layer,
8. stale artifact cleanup,
9. why persisted data creates a new future trust boundary.

## Next: Phase 6

Reliability engineering, broader regression tests, CI, and GitHub Actions.